In [8]:
import pandas as pd

df = pd.read_csv("../data/SwissCrawl.csv")

df

,text,url,crawl_proba,date
0,"Me het dä Chram zsämepackt, und der Papa het w...",https://gutenberg.spiegel.de/buch/der-starn-vo...,0.999427,2019-11-10T18:26:09.053Z
1,Da lähnt sech d'Frou Kätheli ganz fescht a ihr...,https://gutenberg.spiegel.de/buch/der-starn-vo...,0.998864,2019-11-10T18:27:19.476Z
2,"Da isch es aber uus gsi mit der Arbeit. - ""E lue;",https://gutenberg.spiegel.de/buch/der-starn-vo...,0.999498,2019-11-10T18:27:00.879Z
3,Es gwüsses Beduure mit däm bidere treue Meitsc...,https://gutenberg.spiegel.de/buch/der-starn-vo...,0.999690,2019-11-10T18:26:47.910Z
4,"Dert passet er, und schier hätt er no müesse l...",https://gutenberg.spiegel.de/buch/der-starn-vo...,0.999708,2019-11-10T18:26:17.551Z
...,...,...,...,...
563032,hey heckspoiler passt super wie ist die hutze ...,http://www.celica-t23.ch/new/wbb2/thread.php?g...,0.980398,2019-09-29T02:07:36.460Z
563033,äs veilside bodykit chostet ja öppä so viu,http://www.celica-t23.ch/new/wbb2/search.php?s...,0.995973,2019-09-28T11:31:17.908Z
563034,Weiss niemer ä schlauä Usgehtip über d'Feschtt...,http://www.skaladdin.ch/en/guestbookentries.xml,0.998761,2019-09-29T15:59:26.308Z
563035,De Samichlaus het sini schwäre Säck wo gfüllt ...,https://babyglueck.ch/maerchenland/,0.998952,2019-09-29T19:19:41.834Z


In [9]:
df["domain"] = df["url"].str.extract(r"https?://([^/]+)")
df["tld"] = df["domain"].str.split(".").str[-1]

print(df["tld"].value_counts(normalize=True).head(10))
print(df[df["tld"] == "ch"]["domain"].value_counts().head(30))

tld
ch       0.713994
com      0.194795
org      0.042150
de       0.036765
net      0.006744
tips     0.001703
fr       0.001499
info     0.000927
eu       0.000490
rocks    0.000348
Name: proportion, dtype: float64
domain
www.fcbforum.ch              54980
www.heiraten.ch              36145
forum.zscfans.ch             31141
www.celica-t23.ch            26469
www.fczforum.ch              23977
swizzlink.ch                  7467
hitparade.ch                  7298
www.heiner-oberer.ch          6936
www.liamtyler.ch              5973
wettiger-nochrichte.ch        5076
www.schnitzelbankbasel.ch     2920
www.watson.ch                 2783
www.festzeit.ch               2653
www.babyforum.ch              2496
www.porretsch.ch              2414
www.basler-dybli.ch           2366
spale.ch                      2342
www.vopap.ch                  2309
www.skaladdin.ch              2044
www.lams.ch                   1975
www.pascaltschudi.ch          1903
horni-ballonfahrten.ch        1802
www.fr

In [10]:
for d in df["domain"].value_counts().head(40).index:
    print(f"\n=== {d}")
    for t in df[df["domain"] == d]["text"].sample(3, random_state=0):
        print("  ", t[:150])


=== www.fcbforum.ch
   Guete Resultat, die werde uns no lang z Rede geh
   logo und wärde biesst/gstrooft wenn mir s off.
   Au wenn ich nume e dummi schnurre über dich gha han.

=== www.wikiwand.com
   Di einzig Ortschaft, wo mer es no e weng schwätzt, isch Präz.
   Dr Punkt ganz zoberscht isch uf 4634 Meter über Meer.
   Im Jahr 1962 isch de Frisch, wo denn 51-jährig gsi isch, de 23-jährige Studäntin Marianne Oellers begägnet, wo zerscht mit ihm zämmegläbt und ihn schl

=== www.heiraten.ch
   Gniesset jedi sekunde vo eurem dag, wünsch euch nur d best für hüt (und natürli au für suscht) ihr händ das beidi vrdient Bi scho seeehr gspannt uf fö
   Langsam merk i, dass alles echli beschwerlicher wird und ich nüme die schnellsti bin.
   also ich finds nid obligatorisch...glaub das närvt mi nur und rapse am bei.

=== forum.zscfans.ch
   Öppis wo grad de Meier - en Trainer us dere Jugend - fascht konsequänt verweigeret hät mit de Konsequänz, dass "e ganzi Generation" (eini mit überdurc
   a

In [11]:
INCLUDE = {
    "forum": {
        "www.fcbforum.ch", "www.fczforum.ch", "forum.zscfans.ch",
        "www.heiraten.ch", "www.babyforum.ch", "www.celica-t23.ch",
        "swizzlink.ch", "www.festzeit.ch", "www.skaladdin.ch", "hitparade.ch",
    },
    "social": {"www.facebook.com", "twitter.com"},
    "news_comments": {"www.watson.ch", "www.derbund.ch"},
}


GROUP_OF = {d: g for g, ds in INCLUDE.items() for d in ds}

df["source_group"] = df["domain"].map(GROUP_OF)
df_keep = df[df["source_group"].notna()].reset_index(drop=True)

print(len(df), "->", len(df_keep))
print(df_keep["source_group"].value_counts())

563037 -> 225822
source_group
forum            194670
social            26685
news_comments      4467
Name: count, dtype: int64


In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("deepset/gbert-base")

# Language confidence
df_keep = df_keep[df_keep["crawl_proba"] > 0.99]

# Length: drop sentences longer than the max_length used in training (128 tokens)
enc = tokenizer(df_keep["text"].tolist())["input_ids"]
n_tokens = pd.Series([len(x) for x in enc], index=df_keep.index)
df_keep = df_keep[n_tokens <= 128]

# Duplicates (ignoring case and whitespace)
df_keep["norm"] = df_keep["text"].str.lower().str.split().str.join(" ")
df_keep = df_keep.drop_duplicates(subset="norm").drop(columns="norm").reset_index(drop=True)

df_keep["sentence_id"] = range(len(df_keep))

print(len(df_keep))
print(df_keep["source_group"].value_counts())

209487
source_group
forum            181693
social            24309
news_comments      3485
Name: count, dtype: int64


In [13]:
df_keep.to_csv("../data/swisscrawl_filtered.csv", index=False)

In [14]:
sample = df_keep.sample(700, random_state=42)

shared = sample.sample(100, random_state=42)
rest = sample.drop(shared.index).sample(frac=1, random_state=42)

own = {
    "dario": rest.iloc[:200],
    "marjorie": rest.iloc[200:400],
    "yannis": rest.iloc[400:600],
}

for i, (name, part) in enumerate(own.items()):
    f = pd.concat([shared, part]).sample(frac=1, random_state=42 + i)[["sentence_id", "text"]]
    f["label"] = ""
    f["comment"] = ""
    f.to_csv(f"../data/annotation_{name}.csv", index=False)

sample.to_csv("../data/master_sample.csv", index=False)